In [0]:
# Imports needed in this notebook (each notebook has its own memory)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read the monthly balance mart built in the previous step
balance = spark.table("workspace.gold.mart_monthly_balance")

In [0]:
# The window defines the ORDER of the rows: year first, then month
# No partitionBy: we want ONE single timeline with all months
w = Window.orderBy("year", "month")

mom = (
    balance
    # LAG(column, 1) = value of the previous row (1 row above) in the window order
    .withColumn("prev_net_balance", F.lag("net_balance", 1).over(w))
    # Absolute change: this month minus last month
    .withColumn("mom_change", F.col("net_balance") - F.col("prev_net_balance"))
    # Percentage change; abs() in the denominator because the balance can be negative
    .withColumn(
        "mom_change_pct",
        F.round(F.col("mom_change") / F.abs(F.col("prev_net_balance")) * 100, 2)
    )
    .select("year", "month", "month_name", "net_balance",
            "prev_net_balance", "mom_change", "mom_change_pct")
)

mom.show()

In [0]:
#Etapa 9.2: total acumulado (running total)

In [0]:
# Same ordering as before: year first, then month
# rowsBetween defines WHICH rows the SUM looks at:
#   unboundedPreceding = from the very first row
#   currentRow         = up to the row we are on
w_cum = (Window.orderBy("year", "month")
               .rowsBetween(Window.unboundedPreceding, Window.currentRow))

cumulative = (
    balance
    # SUM over the window = sum of all months from the first one up to this one
    .withColumn("cumulative_net_balance", F.sum("net_balance").over(w_cum))
    .select("year", "month", "month_name", "net_balance", "cumulative_net_balance")
    .orderBy("year", "month")
)

cumulative.show()

In [0]:
#Etapa 9.3: ranking de categorias com RANK e DENSE_RANK

In [0]:
# Read the category mart (one row per month + category)
mart_cat = spark.table("workspace.gold.mart_monthly_category")

# partitionBy: ranking restarts for each (year, month), like one leaderboard per month
# orderBy desc: the biggest total_amount gets position 1
w_rank = Window.partitionBy("year", "month").orderBy(F.col("total_amount").desc())

ranked = (
    mart_cat
    # Only spending categories make sense for a "where does the money go" ranking
    .filter(F.col("flow_type") == "outflow")
    # RANK: ties share a position and the next position is skipped (1,2,2,4)
    .withColumn("rank_in_month", F.rank().over(w_rank))
    # DENSE_RANK: ties share a position and the next one is NOT skipped (1,2,2,3)
    .withColumn("dense_rank_in_month", F.dense_rank().over(w_rank))
    .select("year", "month", "month_name", "category_name",
            "total_amount", "rank_in_month", "dense_rank_in_month")
    .orderBy("year", "month", "rank_in_month")
)

ranked.show(20)